# Six routing-selection baselines against TAPOCT + TAPOCB-NN

The two fixed-solvers are **LKH-3** and **MatNet**. The four portfolio selectors are
Degroote et al.'s separate per-arm regression (SoCS 2018, Algorithm 2), NeuralUCB
(Zhou et al., ICML 2020, Algorithm 1), NeuralTS (Zhang et al., ICLR 2021,
Algorithm 1), and NeuralRBMLE-PC (Hung and Hsieh, AAAI 2023, Algorithm 2).
The neural implementations use **full** Jacobian covariance and bounded scalar
client scores. The network itself is trained using minibatch replay every eight
observations instead of exact all-history fitting at every round; report that
approximation and the configured width in the paper.

Two measured calls per arm on each **training polygon** are given to all four
portfolio competitors, matching TAPOCT's minimum measurement budget. Online
updates still use only the selected arm's response. The evaluation polygons are the same grouped
test split used for TAPOCB-NN, and no target preference weights are exposed to
any selector. All methods face the same tender sequence, price, road matrix,
and client response rule. The two single solvers have no warm-start training.


In [35]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import tapoct_fast as tf
import TAPOCT_robustness_cell as rb
import portfolio_sota as ps

PATHS = {
    'map': 'AUG_global_map_osrm_seconds.pkl',
    'properties': 'global_properties_osrm_seconds.pkl',
    'results': 'epos_cache_fd.pkl',
    'couriers': 'courier_training_checkpoint_fd_seconds.pkl',
    'preferences': 'glob_pref_map.pkl',
}
NN_DIR = Path('courier_models_fd_8_scenarios')
OUT = Path('portfolio_sota_comparison')
OUT.mkdir(exist_ok=True)
COURIER_ID = 35  # set to the ID used by your primary TAPOCB-NN experiment
SEED = 42
MAX_REPEATS = 50
# True: one measured solver outcome per polygon/arm is replayed across scenarios.
# False: live solver execution for every tender and every method; much slower.
REUSE_ROUTES = True
ROUTE_DRAWS = 1

loaded = {}
for key, path in PATHS.items():
    with open(path, 'rb') as f:
        loaded[key] = pickle.load(f)
AUG_global_map = loaded['map']
properties_global = loaded['properties']
EPOS_results = loaded['results']
trainer = loaded['couriers']
GLOBAL_PREFERENCES_MAP = loaded['preferences']
if COURIER_ID not in trainer:
    raise KeyError(f'Missing courier {COURIER_ID}; available: {list(trainer)[:10]}')
print('Loaded courier', COURIER_ID)


Loaded courier 35


In [36]:
# Generate only scenario preference maps; no solver is called here.
# This is the same seed/reference as Supportive_NN_integrated.ipynb.
replay = rb.common_triplet_replay(EPOS_results, ps.ALGORITHMS)
scenario_maps, _, diagnostics = rb.preference_maps(
    replay, AUG_global_map, properties_global, seed=SEED,
    gaussian_reference=GLOBAL_PREFERENCES_MAP)

courier = trainer[COURIER_ID]
ids = [p for p in dict.fromkeys(courier['polygon_list'])
       if p in replay and p in AUG_global_map and p in properties_global]
# The existing MatNet has embed_dim=128 and creates one-hot node embeddings;
# matrices above 128 nodes cannot be evaluated by this checkpoint.
train_ids, val_ids, test_ids = ps.grouped_partition(ids, AUG_global_map, SEED)
excluded_oversize = sum(len(AUG_global_map[p]['full map']) + 1 > 128 for p in test_ids)
test_ids = [p for p in test_ids if len(AUG_global_map[p]['full map']) + 1 <= 128]
sequence = ps.sequence_with_repetitions(test_ids, properties_global,
                                         seed=SEED, max_repeats=MAX_REPEATS)
scaler = ps.build_context_scaler(train_ids, properties_global, AUG_global_map)
assert set(train_ids).isdisjoint(test_ids)
print('Coverage:', len(ids), 'train:', len(train_ids), 'val:', len(val_ids),
      'test:', len(test_ids), 'evaluation tender count:', len(sequence),
      'excluded >128-node polygons:', excluded_oversize)


Coverage: 3850 train: 2460 val: 620 test: 770 evaluation tender count: 16760 excluded >128-node polygons: 0


In [37]:
# Build depot->order rows once. This is the same lookup direction as
# depot_row_fn in the main notebook: get_distance(array, index, 0, order_id).
# Prefer stored OSRM depot rows, if present; use the operational index otherwise.
DEPOT_POINT = 0
if any(AUG_global_map[p].get('depot row') is None for p in test_ids):
    if 'distance_array' not in globals() or 'idx_dict' not in globals():
        # A view onto distances.dat avoids reading the entire large file into RAM.
        distance_array = np.memmap('distances.dat', dtype=np.int32, mode='r').reshape(-1, 3)
        source_ids = distance_array[:, 0]
        begin = int(np.searchsorted(source_ids, DEPOT_POINT, side='left'))
        end = int(np.searchsorted(source_ids, DEPOT_POINT, side='right'))
        idx_dict = {DEPOT_POINT: (begin, end)}
        if begin == end:
            raise KeyError(f'Missing depot source {DEPOT_POINT} in distances.dat')
    if 'get_distance' not in globals():
        def get_distance(distance_array, idx_dict, from_point, to_point, default=10**9):
            if from_point not in idx_dict:
                return default
            begin, end = idx_dict[from_point]
            subset = distance_array[begin:end]
            pos = int(np.searchsorted(subset[:, 1], to_point))
            return subset[pos, 2] if pos < len(subset) and subset[pos, 1] == to_point else default

def from_depot_row(full_map_ids):
    return [get_distance(distance_array, idx_dict, DEPOT_POINT,
                         int(str(order).rsplit('-', 1)[-1])) for order in full_map_ids]

depot_rows = ps.precompute_depot_rows(test_ids, AUG_global_map, from_depot_row)
stored_rows = sum(AUG_global_map[p].get('depot row') is not None for p in test_ids)
print('Precomputed', len(depot_rows), 'depot rows in seconds:',
      stored_rows, 'stored OSRM;', len(test_ids)-stored_rows, 'operational-index fallback')


Precomputed 770 depot rows in seconds: 693 stored OSRM; 77 operational-index fallback


In [38]:
# Fit four scenario-specific portfolio selectors.
# Targets for these selectors are OFFER satisfaction (feedback), not the
# TAPOCT probability vector from the teacher. The client weights stay hidden.
SCENARIOS = [(mode, dist) for mode in rb.SPATIAL_MODES for dist in rb.SCENARIOS]
checkpoint_paths = {}
training_log = []
for mode, distribution in SCENARIOS:
    scenario = (mode, distribution)
    prefs = scenario_maps[mode][distribution]
    logged = ps.make_logged_feedback(train_ids, EPOS_results, prefs,
                                     seed=SEED, calls_per_arm=2)
    selectors, scaler_s = ps.train_logged_selectors(
        logged, properties_global, AUG_global_map, seed=SEED, scaler=scaler,
        pretrain_epochs=5, width=16, ridge=1.0, exploration=.15,
        train_every=8, gradient_steps=2, batch_size=32)
    assert np.allclose(scaler.mean_, scaler_s.mean_)
    path = OUT / f'baselines_{COURIER_ID}_{mode}_{distribution}.pth'
    # Checkpoint includes precisely the information the selectors may know.
    torch.save({'scenario': scenario, 'courier_id': COURIER_ID,
                'train_ids': train_ids, 'val_ids': val_ids, 'test_ids': test_ids,
                'scaler': scaler, 'logged_arm_counts': np.bincount(
                    [k for _, k, _ in logged], minlength=4),
                'regression': selectors['Regression'],
                'neural': {name: selectors[name].export()
                           for name in ps.COMPETITORS[3:]}}, path)
    checkpoint_paths[scenario] = path
    training_log.append({'spatial_mode': mode, 'distribution': distribution,
                         'n_logged': len(logged), 'checkpoint': str(path)})
    print('Trained', scenario, len(logged), 'measured training observations')
pd.DataFrame(training_log).to_csv(OUT / 'training_manifest.csv', index=False)


Trained ('independent', 'gaussian') 19680 measured training observations
Trained ('independent', 'beta') 19680 measured training observations
Trained ('independent', 'dirichlet') 19680 measured training observations
Trained ('independent', 'ordinal_posterior') 19680 measured training observations
Trained ('spatial', 'gaussian') 19680 measured training observations
Trained ('spatial', 'beta') 19680 measured training observations
Trained ('spatial', 'dirichlet') 19680 measured training observations
Trained ('spatial', 'ordinal_posterior') 19680 measured training observations


In [39]:
# Optional: evaluate a short paired smoke run before the full sensitivity sweep.
# routing_solvers eagerly loads the saved MatNet and NLNS weights; paths in
# routing_solvers.py must point to the actual .pth files. It is imported once.
import routing_solvers
solvers = routing_solvers.get_solvers()
ps.warmup_solver_bundle(solvers, test_ids[0], AUG_global_map, depot_rows=depot_rows)

def load_warmstart(scenario):
    saved = torch.load(checkpoint_paths[scenario], map_location='cpu', weights_only=False)
    assert saved['scenario'] == scenario and saved['test_ids'] == test_ids
    return ({'Regression': saved['regression'],
             **{name: ps.NeuralSelector.restore(state, seed=SEED)
                for name, state in saved['neural'].items()}}, saved['scaler'])

# The scenario and polygon are fixed explicitly for a quick integration check.
smoke_scenario = ('spatial', 'gaussian')
warmstart, local_scaler = load_warmstart(smoke_scenario)
smoke = ps.run_scenario(
    sequence[:3], smoke_scenario, aug_map=AUG_global_map,
    properties=properties_global, epos_results=EPOS_results,
    preference_map=scenario_maps[smoke_scenario[0]][smoke_scenario[1]],
    tapocb_checkpoint=NN_DIR / f'courier_{COURIER_ID}_{smoke_scenario[0]}_{smoke_scenario[1]}.pth',
    warmstart_selectors=warmstart, context_scaler=local_scaler,
    solvers=solvers, seed=SEED, reuse_routes=REUSE_ROUTES,
    route_draws=ROUTE_DRAWS, depot_rows=depot_rows)
assert smoke.groupby('step')['method'].nunique().eq(7).all()
display(smoke[['step', 'method', 'arm', 'route_cost_s', 'total_ct_s', 'client_reward']])


,step,method,arm,route_cost_s,total_ct_s,client_reward
0,0,TAPOCB-NN,Christofides,5630.0,0.021557,0.841202
1,0,LKH-3,lkh3,5296.0,0.039702,0.839683
2,0,MatNet,MatNet,8532.0,0.040376,0.000000
3,0,Regression,Christofides,5630.0,0.020923,0.841295
4,0,NeuralUCB,Christofides,5630.0,0.022504,0.841064
5,0,NeuralTS,Christofides,5630.0,0.022290,0.841095
6,0,NeuralRBMLE-PC,Christofides,5630.0,0.023695,0.840890
7,1,TAPOCB-NN,Christofides,3906.0,0.004387,0.838569
8,1,LKH-3,lkh3,3880.0,0.031815,0.835780
9,1,MatNet,MatNet,4356.0,0.025531,0.817067


In [40]:
# Full eight-scenario sweep. Rerun this after updating portfolio_sota.py:
# the full TAPOCB trajectory must use the same rho/gamma rule as all ablations.
import importlib
ps = importlib.reload(ps)
# route_cache stores exogenous, measured solver (cost, CT) outcomes only;
# it does not contain client preferences or feedback, and is shared across
# scenarios to make the analysis feasible within the revision deadline.
import json
POLICY_RULE_MARKER = OUT / 'tapocb_policy_rule.json'
POLICY_RULE_MARKER.unlink(missing_ok=True)  # prevent an interrupted rerun from looking complete
route_cache = {} if REUSE_ROUTES else None
raw_files = []
for mode, distribution in SCENARIOS:
    scenario = (mode, distribution)
    warmstart, local_scaler = load_warmstart(scenario)
    nn_path = NN_DIR / f'courier_{COURIER_ID}_{mode}_{distribution}.pth'
    if not nn_path.exists():
        raise FileNotFoundError(f'Missing trained TAPOCB-NN model: {nn_path}')
    raw = ps.run_scenario(
        sequence, scenario, aug_map=AUG_global_map, properties=properties_global,
        epos_results=EPOS_results, preference_map=scenario_maps[mode][distribution],
        tapocb_checkpoint=nn_path, warmstart_selectors=warmstart,
        context_scaler=local_scaler, solvers=solvers, seed=SEED,
        reuse_routes=REUSE_ROUTES, route_draws=ROUTE_DRAWS,
        route_cache=route_cache, depot_rows=depot_rows)
    path = OUT / f'raw_{mode}_{distribution}.csv'
    raw.to_csv(path, index=False)
    raw_files.append(path)
    print('Completed:', scenario, 'paired tenders:', raw['step'].nunique())
    del raw

POLICY_RULE_MARKER.write_text(json.dumps({
    'version': ps.TAPOCB_POLICY_RULE_VERSION, 'scenarios': [list(s) for s in SCENARIOS],
    'rho': .15, 'feedback_indicator': 'client_score_at_least_LKH3',
    'seed': SEED, 'raw_files': [str(p) for p in raw_files],
}, indent=2), encoding='utf-8')


Completed: ('independent', 'gaussian') paired tenders: 16760
Completed: ('independent', 'beta') paired tenders: 16760
Completed: ('independent', 'dirichlet') paired tenders: 16760
Completed: ('independent', 'ordinal_posterior') paired tenders: 16760
Completed: ('spatial', 'gaussian') paired tenders: 16760
Completed: ('spatial', 'beta') paired tenders: 16760
Completed: ('spatial', 'dirichlet') paired tenders: 16760
Completed: ('spatial', 'ordinal_posterior') paired tenders: 16760


1071

In [41]:
# Full sensitivity tables. Intervals use paired, original-polygon block bootstrap.
# Reload the updated aggregator when running this cell in an existing notebook kernel.
import importlib
ps = importlib.reload(ps)
# Compact report: mean and median intervals only for TAPOCB-NN and the best SOTA.
# top_1 is selected by the accumulated MEAN reward, then its median is reported.
CI_BOOTSTRAP = 1000
CI_LEVEL = .95
CLUSTER_BASE_COLUMNS = [
    'scenario', 'cluster', 'n_tenders', 'n_original_polygons',
    'avg_TAPOCB-NN_reward', 'avg_TAPOCB-NN_reward_ci_low',
    'avg_TAPOCB-NN_reward_ci_high',
    'median_TAPOCB-NN_reward', 'median_TAPOCB-NN_reward_ci_low',
    'median_TAPOCB-NN_reward_ci_high',
    'top_1',
    'avg_top_1_reward', 'avg_top_1_reward_ci_low', 'avg_top_1_reward_ci_high',
    'median_top_1_reward', 'median_top_1_reward_ci_low',
    'median_top_1_reward_ci_high',
    'avg_delta_TAPOCB-NN_from_top_1_units',
    'avg_delta_TAPOCB-NN_from_top_1_share',
    'median_delta_TAPOCB-NN_from_top_1_units',
    'median_delta_TAPOCB-NN_from_top_1_share',
    'TAPOCB-NN_win_rate',
]
CLUSTER_ABLATION_COLUMNS = CLUSTER_BASE_COLUMNS + [
    field for method in ps.COMPETITORS for field in (
        f'avg_{method}_reward',
        f'median_{method}_reward',
        f'delta_TAPOCB_from_{method}_units',
        f'delta_TAPOCB_from_{method}_share')]
assert len(CLUSTER_ABLATION_COLUMNS) == 46
SCENARIO_ABLATION_COLUMNS = [c for c in CLUSTER_ABLATION_COLUMNS if c != 'cluster']

raw_all = pd.concat([pd.read_csv(p) for p in raw_files], ignore_index=True)
by_cluster, by_scenario = ps.summarize_sensitivity(
    raw_all, aug_map=AUG_global_map, bootstrap_samples=CI_BOOTSTRAP,
    ci_level=CI_LEVEL, seed=SEED)
by_cluster = by_cluster[CLUSTER_ABLATION_COLUMNS].sort_values(
    ['scenario', 'cluster']).reset_index(drop=True)
by_scenario = by_scenario[SCENARIO_ABLATION_COLUMNS].sort_values(
    'scenario').reset_index(drop=True)
assert by_cluster.shape == (40, 46) and by_scenario.shape == (8, 45)
by_cluster.to_csv(OUT / 'sensitivity_by_cluster.csv', index=False)
by_scenario.to_csv(OUT / 'sensitivity_by_scenario.csv', index=False)
display(by_scenario[['scenario', 'n_tenders', 'n_original_polygons',
                     'avg_TAPOCB-NN_reward', 'avg_TAPOCB-NN_reward_ci_low',
                     'avg_TAPOCB-NN_reward_ci_high',
                     'median_TAPOCB-NN_reward', 'median_TAPOCB-NN_reward_ci_low',
                     'median_TAPOCB-NN_reward_ci_high',
                     'top_1', 'avg_top_1_reward',
                     'median_top_1_reward', 'median_top_1_reward_ci_low',
                     'median_top_1_reward_ci_high',
                     'avg_delta_TAPOCB-NN_from_top_1_units',
                     'median_delta_TAPOCB-NN_from_top_1_units',
                     'TAPOCB-NN_win_rate']])
display(by_cluster)
display(by_scenario[['scenario', 'median_TAPOCB-NN_reward'] +
                    [f'median_{name}_reward' for name in ps.COMPETITORS]])
# The CSVs also contain median_{method}_reward for all six SOTA methods.


,scenario,n_tenders,n_original_polygons,avg_TAPOCB-NN_reward,avg_TAPOCB-NN_reward_ci_low,avg_TAPOCB-NN_reward_ci_high,median_TAPOCB-NN_reward,median_TAPOCB-NN_reward_ci_low,median_TAPOCB-NN_reward_ci_high,top_1,avg_top_1_reward,median_top_1_reward,median_top_1_reward_ci_low,median_top_1_reward_ci_high,avg_delta_TAPOCB-NN_from_top_1_units,median_delta_TAPOCB-NN_from_top_1_units,TAPOCB-NN_win_rate
0,independent/beta,16760,77,0.587629,0.550241,0.619396,0.653785,0.640846,0.667006,LKH-3,0.595690,0.655769,0.642102,0.666717,-0.008060,-0.001984,0.705400
1,independent/dirichlet,16760,77,0.596303,0.559740,0.630334,0.658571,0.643092,0.670980,LKH-3,0.604658,0.659323,0.644701,0.671740,-0.008355,-0.000752,0.699911
2,independent/gaussian,16760,77,0.593774,0.553457,0.626719,0.662304,0.640477,0.679353,LKH-3,0.602731,0.664802,0.644456,0.680326,-0.008957,-0.002498,0.710591
3,independent/ordinal_posterior,16760,77,0.573503,0.536740,0.606081,0.639015,0.611970,0.651880,LKH-3,0.589194,0.641333,0.620892,0.657211,-0.015691,-0.002318,0.675805
4,spatial/beta,16760,77,0.615578,0.567752,0.655234,0.688276,0.655250,0.704905,LKH-3,0.623383,0.688906,0.657682,0.705117,-0.007805,-0.000631,0.723001
5,spatial/dirichlet,16760,77,0.612728,0.565236,0.651387,0.682070,0.652568,0.699370,LKH-3,0.620719,0.682254,0.653838,0.698349,-0.007991,-0.000184,0.720973
6,spatial/gaussian,16760,77,0.622814,0.582286,0.661435,0.699620,0.682516,0.713670,LKH-3,0.630903,0.699999,0.683012,0.712628,-0.008089,-0.000379,0.734636
7,spatial/ordinal_posterior,16760,77,0.608689,0.566525,0.648772,0.676406,0.648104,0.697887,LKH-3,0.623475,0.680806,0.657407,0.704194,-0.014786,-0.004400,0.699433


,scenario,cluster,n_tenders,n_original_polygons,avg_TAPOCB-NN_reward,avg_TAPOCB-NN_reward_ci_low,avg_TAPOCB-NN_reward_ci_high,median_TAPOCB-NN_reward,median_TAPOCB-NN_reward_ci_low,median_TAPOCB-NN_reward_ci_high,...,delta_TAPOCB_from_NeuralUCB_units,delta_TAPOCB_from_NeuralUCB_share,avg_NeuralTS_reward,median_NeuralTS_reward,delta_TAPOCB_from_NeuralTS_units,delta_TAPOCB_from_NeuralTS_share,avg_NeuralRBMLE-PC_reward,median_NeuralRBMLE-PC_reward,delta_TAPOCB_from_NeuralRBMLE-PC_units,delta_TAPOCB_from_NeuralRBMLE-PC_share
0,independent/beta,0,739,13,0.205811,0.089937,0.338259,0.000000,0.000000,0.517858,...,-0.015695,-0.070856,0.223884,0.000000,-0.018072,-0.080721,0.185039,0.000000,0.020772,0.112259
1,independent/beta,2,12665,51,0.660114,0.647677,0.670557,0.671692,0.659953,0.685924,...,-0.003936,-0.005927,0.663607,0.672154,-0.003493,-0.005263,0.659555,0.667439,0.000560,0.000849
2,independent/beta,3,68,2,0.061952,0.000000,0.091581,0.000000,0.000000,0.000000,...,-0.001631,-0.025656,0.082167,0.000000,-0.020215,-0.246020,0.044498,0.000000,0.017454,0.392235
3,independent/beta,5,2439,24,0.440488,0.346850,0.559202,0.591184,0.501085,0.650021,...,-0.010743,-0.023808,0.450303,0.591028,-0.009815,-0.021797,0.447186,0.588933,-0.006699,-0.014980
4,independent/beta,7,849,13,0.303490,0.225733,0.386481,0.421585,0.000000,0.543177,...,-0.014333,-0.045096,0.310914,0.452977,-0.007424,-0.023877,0.292208,0.386309,0.011282,0.038610
5,independent/dirichlet,0,739,13,0.209589,0.101661,0.338815,0.000000,0.000000,0.521975,...,-0.018837,-0.082463,0.218019,0.000000,-0.008430,-0.038666,0.194749,0.000000,0.014840,0.076200
6,independent/dirichlet,2,12665,51,0.669969,0.657415,0.682622,0.673193,0.661572,0.687430,...,-0.004015,-0.005957,0.673391,0.673483,-0.003422,-0.005081,0.672758,0.671751,-0.002789,-0.004146
7,independent/dirichlet,3,68,2,0.075296,0.000000,0.111307,0.000000,0.000000,0.000000,...,0.008787,0.132110,0.084208,0.000000,-0.008912,-0.105835,0.045963,0.000000,0.029334,0.638208
8,independent/dirichlet,5,2439,24,0.443860,0.351232,0.544118,0.579215,0.491958,0.626049,...,-0.013522,-0.029564,0.455569,0.587933,-0.011709,-0.025702,0.450742,0.578300,-0.006882,-0.015268
9,independent/dirichlet,7,849,13,0.313654,0.221594,0.400478,0.324526,0.000000,0.529480,...,0.001889,0.006060,0.323744,0.326203,-0.010090,-0.031167,0.303804,0.354942,0.009851,0.032424


,scenario,median_TAPOCB-NN_reward,median_LKH-3_reward,median_MatNet_reward,median_Regression_reward,median_NeuralUCB_reward,median_NeuralTS_reward,median_NeuralRBMLE-PC_reward
0,independent/beta,0.653785,0.655769,0.572705,0.654207,0.654033,0.654199,0.650736
1,independent/dirichlet,0.658571,0.659323,0.568511,0.659219,0.657750,0.658409,0.656457
2,independent/gaussian,0.662304,0.664802,0.551736,0.663704,0.662703,0.661913,0.659481
3,independent/ordinal_posterior,0.639015,0.641333,0.533946,0.642859,0.643210,0.640484,0.638240
4,spatial/beta,0.688276,0.688906,0.605669,0.690699,0.688780,0.688958,0.683808
5,spatial/dirichlet,0.682070,0.682254,0.584601,0.684551,0.681798,0.682539,0.678034
6,spatial/gaussian,0.699620,0.699999,0.634624,0.701106,0.699504,0.699472,0.690812
7,spatial/ordinal_posterior,0.676406,0.680806,0.574310,0.682157,0.680850,0.679868,0.675947


In [42]:
# Seven ablations: no NN warm-start, four solver exclusions, no online learning,
# and deterministic online adaptation without a Dirichlet injection.
# Run after the full sweep and its raw_all table (cells 1-8 above), without
# changing solver weights or restarting the kernel. The six SOTA policies are
# independent of TAPOCB-NN, so their measured trajectories can be reused.
ABLATION_DIR = OUT / 'three_component_ablations'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
if not REUSE_ROUTES or not isinstance(route_cache, dict) or not route_cache:
    raise RuntimeError('Run the full paired sweep with REUSE_ROUTES=True before ablations')
if not POLICY_RULE_MARKER.exists():
    raise RuntimeError('Rerun the full sweep (cell 7) under the revised rho/gamma rule')
policy_rule = json.loads(POLICY_RULE_MARKER.read_text(encoding='utf-8'))
if (policy_rule.get('version') != ps.TAPOCB_POLICY_RULE_VERSION
        or policy_rule.get('scenarios') != [list(s) for s in SCENARIOS]):
    raise RuntimeError('Full-policy baseline was generated under a different policy rule')

competitor_rows = raw_all[raw_all['method'].isin(ps.COMPETITORS)].copy()
expected_steps = pd.DataFrame({'step': np.arange(len(sequence)), 'polygon_id': sequence})
expected_scenarios = {'/'.join(s) for s in SCENARIOS}
if set(competitor_rows['scenario']) != expected_scenarios:
    raise ValueError('SOTA replay is missing or includes unexpected scenarios')
for scenario_name, frame in competitor_rows.groupby('scenario'):
    observed = frame.groupby('step').agg(polygon_id=('polygon_id', 'first'),
                                         n_methods=('method', 'nunique'),
                                         n_rows=('method', 'size')).reset_index()
    if (not observed[['step', 'polygon_id']].equals(expected_steps)
            or not observed['n_methods'].eq(len(ps.COMPETITORS)).all()
            or not observed['n_rows'].eq(len(ps.COMPETITORS)).all()):
        raise ValueError(f'{scenario_name}: SOTA replay is not complete and paired')


# TAPOCBMemory in portfolio_sota.py implements Eq. (18)-(19) for both the
# full model and every online ablation. Do not duplicate its update here.


def run_tapocb_variant(sequence, scenario, kind, lkh_reference, *,
                       model, preference_map, cache, disabled_arm=None):
    """Re-evaluate only the target policy; SOTA rows come from paired replay.

    no_warmstart: uniform prior + original online memory (no NN inference).
    drop_arm: NN prior + online memory, with one arm removed throughout.
    no_online: NN prior only; no feedback access or memory updates.
    no_injection: NN prior + cumulative deterministic feedback update, rho gate off.
    """
    if kind not in ('no_warmstart', 'drop_arm', 'no_online', 'no_injection'):
        raise ValueError(f'Unknown ablation: {kind}')
    if (kind == 'drop_arm') != (disabled_arm is not None):
        raise ValueError('A disabled arm is required only for drop_arm')
    if kind != 'no_warmstart' and model is None:
        raise ValueError(f'{kind}: expected scenario-specific NN checkpoint')
    action_seed, exploration_seed = np.random.SeedSequence(SEED).spawn(2)
    rng = np.random.default_rng(action_seed)
    exploration_rng = np.random.default_rng(exploration_seed)
    if kind == 'no_online':
        memory = None
    else:
        enabled = np.ones(len(ps.ALGORITHMS), dtype=bool)
        if kind == 'drop_arm':
            enabled[disabled_arm] = False
        memory = ps.TAPOCBMemory(allowed_arms=enabled,
                                 injection_enabled=(kind != 'no_injection'))
    name = '/'.join(scenario)
    rows = []
    import time
    for step, pid in enumerate(sequence):
        if pid not in preference_map:
            raise KeyError(f'{name}/{pid}: missing client preference')
        spec = ps._spec_for(pid, EPOS_results)
        oracle = tf.ClientFeedbackOracle(preference_map[pid], spec)
        t0 = time.perf_counter()
        if kind == 'no_warmstart':
            prior = np.full(len(ps.ALGORITHMS), 1.0 / len(ps.ALGORITHMS))
        else:
            raw_x = ps.features_for_polygon(pid, properties_global, AUG_global_map)
            prior = model.predict(raw_x)
        if kind == 'no_online':
            p = np.asarray(prior, dtype=float)
            if not np.isfinite(p).all() or (p < 0).any() or p.sum() <= 0:
                raise ValueError(f'{name}/{pid}: invalid NN-only policy')
            p = p / p.sum()
        else:
            p = memory.get_policy(pid, prior, exploration_rng)
        if kind == 'drop_arm' and p[disabled_arm] != 0:
            raise AssertionError(f'{name}: disabled arm acquired probability')
        if not np.isclose(p.sum(), 1, atol=1e-12):
            raise AssertionError(f'{name}: policy does not sum to one')
        arm = int(rng.choice(len(ps.ALGORITHMS), p=p))
        selection_ct = time.perf_counter() - t0
        cost, solver_ct = ps.measure_route(
            pid, arm, aug_map=AUG_global_map, solvers=solvers, cache=cache,
            repeat_index=step % ROUTE_DRAWS, depot_rows=depot_rows)
        reward = float(oracle(spec.price, cost, solver_ct + selection_ct))
        if memory is not None:
            reference_pid, reference_reward = lkh_reference[step]
            if pid != reference_pid:
                raise AssertionError(f'{name}/step {step}: LKH replay is not paired')
            memory.record_outcome(pid, arm, reward, reward >= reference_reward)
        rows.append({'scenario': name, 'step': step, 'polygon_id': pid,
                     'cluster': AUG_global_map[pid]['cluster'],
                     'method': 'TAPOCB-NN', 'arm': ps.ALGORITHMS[arm],
                     'route_cost_s': cost, 'solver_ct_s': solver_ct,
                     'selection_ct_s': selection_ct,
                     'total_ct_s': solver_ct + selection_ct,
                     'client_reward': reward,
                     'rho': memory.rho if memory is not None else np.nan,
                     'gamma_t': (memory.entries[pid]['last_gamma']
                                 if memory is not None else np.nan),
                     'injection_weight': (memory.entries[pid]['last_injection_weight']
                                          if memory is not None else np.nan),
                     'injection_applied': (memory.entries[pid]['last_injection_applied']
                                           if memory is not None else False)})
    return pd.DataFrame(rows), getattr(memory, 'uniform_fallbacks', 0)


# Existing POLICIES schema retains TAPOCB-NN as the target slot; the contour
# and disabled component are identified by the per-table filename and manifest.
ABLATION_VARIANTS = [('no_warmstart', None)] + [
    ('drop_arm', arm) for arm in range(len(ps.ALGORITHMS))
] + [('no_online', None), ('no_injection', None)]
ablation_overview = []
ablation_tables = {}
for kind, disabled_arm in ABLATION_VARIANTS:
    label = ('without_' + ps.ALGORITHMS[disabled_arm].lower().replace('-', '_')
             if kind == 'drop_arm' else kind)
    ablation_raw_parts = []
    for scenario in SCENARIOS:
        name = '/'.join(scenario)
        prior_rows = competitor_rows[competitor_rows['scenario'] == name]
        lkh_rows = prior_rows[prior_rows['method'] == 'LKH-3'].sort_values('step')
        reference = list(zip(lkh_rows['polygon_id'], lkh_rows['client_reward']))
        if len(reference) != len(sequence):
            raise ValueError(f'{name}: missing LKH reference feedback')
        model = None
        if kind != 'no_warmstart':
            checkpoint = NN_DIR / f'courier_{COURIER_ID}_{scenario[0]}_{scenario[1]}.pth'
            if not checkpoint.exists():
                raise FileNotFoundError(checkpoint)
            model = ps.CourierPolicyNetwork(checkpoint)
            if model.scenario != tuple(scenario):
                raise ValueError(f'{checkpoint}: scenario mismatch')
        cached_before = len(route_cache)
        ablated_raw, fallbacks = run_tapocb_variant(
            sequence, scenario, kind, reference, model=model,
            preference_map=scenario_maps[scenario[0]][scenario[1]],
            cache=route_cache, disabled_arm=disabled_arm)
        ablation_raw_parts.append(ablated_raw)
        ablation_overview.append({
            'contour': kind, 'variant': label,
            'disabled_algorithm': (ps.ALGORITHMS[disabled_arm]
                                   if disabled_arm is not None else None),
            'neural_warmstart': kind != 'no_warmstart',
            'online_feedback': kind != 'no_online',
            'dirichlet_injection': kind not in ('no_online', 'no_injection'),
            'scenario': name, 'n_uniform_fallbacks': fallbacks,
            'new_route_measurements': len(route_cache) - cached_before,
        })
        print('Completed ablation:', label, name,
              '| fallback priors:', fallbacks)
    ablated_all = pd.concat(ablation_raw_parts, ignore_index=True)
    paired = pd.concat([competitor_rows, ablated_all], ignore_index=True)
    by_cluster_ablated, by_scenario_ablated = ps.summarize_sensitivity(
        paired, aug_map=AUG_global_map, bootstrap_samples=CI_BOOTSTRAP,
        ci_level=CI_LEVEL, seed=SEED)
    by_cluster_ablated = by_cluster_ablated[CLUSTER_ABLATION_COLUMNS].sort_values(
        ['scenario', 'cluster']).reset_index(drop=True)
    by_scenario_ablated = by_scenario_ablated[SCENARIO_ABLATION_COLUMNS].sort_values(
        'scenario').reset_index(drop=True)
    if by_cluster_ablated.shape != (40, 46) or by_scenario_ablated.shape != (8, 45):
        raise AssertionError(f'{label}: unexpected table dimensions '
                             f'{by_cluster_ablated.shape}, {by_scenario_ablated.shape}')
    ablated_all.to_csv(ABLATION_DIR / f'raw_{label}.csv', index=False)
    by_cluster_ablated.to_csv(ABLATION_DIR / f'cluster_{label}.csv', index=False)
    by_scenario_ablated.to_csv(ABLATION_DIR / f'scenario_{label}.csv', index=False)
    ablation_tables[label] = (by_cluster_ablated, by_scenario_ablated)
    display(by_scenario_ablated[['scenario', 'n_tenders', 'n_original_polygons',
                                 'avg_TAPOCB-NN_reward',
                                 'median_TAPOCB-NN_reward',
                                 'median_TAPOCB-NN_reward_ci_low',
                                 'median_TAPOCB-NN_reward_ci_high',
                                 'top_1', 'avg_top_1_reward',
                                 'median_top_1_reward',
                                 'median_top_1_reward_ci_low',
                                 'median_top_1_reward_ci_high',
                                 'avg_delta_TAPOCB-NN_from_top_1_units',
                                 'median_delta_TAPOCB-NN_from_top_1_units',
                                 'TAPOCB-NN_win_rate']])
    display(by_scenario_ablated[['scenario', 'median_TAPOCB-NN_reward'] +
                                 [f'median_{name}_reward' for name in ps.COMPETITORS]])

pd.DataFrame(ablation_overview).to_csv(ABLATION_DIR / 'run_manifest.csv', index=False)
# Paired change relative to the full four-arm policy (same scenarios and tenders).
full_tapocb = raw_all[raw_all['method'] == 'TAPOCB-NN'][
    ['scenario', 'step', 'polygon_id', 'client_reward']].rename(
        columns={'client_reward': 'full_policy_reward'})
comparison = []
for kind, disabled_arm in ABLATION_VARIANTS:
    label = ('without_' + ps.ALGORITHMS[disabled_arm].lower().replace('-', '_')
             if kind == 'drop_arm' else kind)
    candidate = pd.read_csv(ABLATION_DIR / f'raw_{label}.csv')
    joined = candidate.merge(full_tapocb, on=['scenario', 'step', 'polygon_id'],
                             how='inner', validate='one_to_one')
    if len(joined) != len(sequence) * len(SCENARIOS):
        raise ValueError(f'{label}: missing paired full-policy observations')
    for scenario_name, frame in joined.groupby('scenario'):
        delta = frame['client_reward'] - frame['full_policy_reward']
        comparison.append({'contour': kind, 'variant': label, 'scenario': scenario_name,
                           'n_tenders': len(frame), 'mean_delta_reward': delta.mean(),
                           'median_delta_reward': delta.median(),
                           'ablation_pairwise_win_rate': (delta.gt(0).sum() +
                                                         .5 * delta.eq(0).sum()) / len(delta)})
ablation_vs_full = pd.DataFrame(comparison)
ablation_vs_full.to_csv(ABLATION_DIR / 'ablation_vs_full.csv', index=False)
display(ablation_vs_full)


Completed ablation: no_warmstart independent/gaussian | fallback priors: 0
Completed ablation: no_warmstart independent/beta | fallback priors: 0
Completed ablation: no_warmstart independent/dirichlet | fallback priors: 0
Completed ablation: no_warmstart independent/ordinal_posterior | fallback priors: 0
Completed ablation: no_warmstart spatial/gaussian | fallback priors: 0
Completed ablation: no_warmstart spatial/beta | fallback priors: 0
Completed ablation: no_warmstart spatial/dirichlet | fallback priors: 0
Completed ablation: no_warmstart spatial/ordinal_posterior | fallback priors: 0


,scenario,n_tenders,n_original_polygons,avg_TAPOCB-NN_reward,median_TAPOCB-NN_reward,median_TAPOCB-NN_reward_ci_low,median_TAPOCB-NN_reward_ci_high,top_1,avg_top_1_reward,median_top_1_reward,median_top_1_reward_ci_low,median_top_1_reward_ci_high,avg_delta_TAPOCB-NN_from_top_1_units,median_delta_TAPOCB-NN_from_top_1_units,TAPOCB-NN_win_rate
0,independent/beta,16760,77,0.562385,0.641717,0.628045,0.656687,LKH-3,0.595690,0.655769,0.642102,0.666717,-0.033305,-0.014052,0.630519
1,independent/dirichlet,16760,77,0.570783,0.645811,0.631226,0.661328,LKH-3,0.604658,0.659323,0.644701,0.671740,-0.033875,-0.013512,0.627237
2,independent/gaussian,16760,77,0.571383,0.648047,0.627607,0.666753,LKH-3,0.602731,0.664802,0.644456,0.680326,-0.031348,-0.016755,0.637321
3,independent/ordinal_posterior,16760,77,0.556392,0.626076,0.604051,0.647446,LKH-3,0.589194,0.641333,0.620892,0.657211,-0.032801,-0.015257,0.618825
4,spatial/beta,16760,77,0.590034,0.670127,0.638157,0.693559,LKH-3,0.623383,0.688906,0.657682,0.705117,-0.033350,-0.018780,0.642035
5,spatial/dirichlet,16760,77,0.587558,0.663944,0.635507,0.686722,LKH-3,0.620719,0.682254,0.653838,0.698349,-0.033161,-0.018309,0.641140
6,spatial/gaussian,16760,77,0.598988,0.687754,0.664480,0.705251,LKH-3,0.630903,0.699999,0.683012,0.712628,-0.031916,-0.012245,0.649672
7,spatial/ordinal_posterior,16760,77,0.591773,0.666526,0.638964,0.690982,LKH-3,0.623475,0.680806,0.657407,0.704194,-0.031702,-0.014281,0.646808


,scenario,median_TAPOCB-NN_reward,median_LKH-3_reward,median_MatNet_reward,median_Regression_reward,median_NeuralUCB_reward,median_NeuralTS_reward,median_NeuralRBMLE-PC_reward
0,independent/beta,0.641717,0.655769,0.572705,0.654207,0.654033,0.654199,0.650736
1,independent/dirichlet,0.645811,0.659323,0.568511,0.659219,0.657750,0.658409,0.656457
2,independent/gaussian,0.648047,0.664802,0.551736,0.663704,0.662703,0.661913,0.659481
3,independent/ordinal_posterior,0.626076,0.641333,0.533946,0.642859,0.643210,0.640484,0.638240
4,spatial/beta,0.670127,0.688906,0.605669,0.690699,0.688780,0.688958,0.683808
5,spatial/dirichlet,0.663944,0.682254,0.584601,0.684551,0.681798,0.682539,0.678034
6,spatial/gaussian,0.687754,0.699999,0.634624,0.701106,0.699504,0.699472,0.690812
7,spatial/ordinal_posterior,0.666526,0.680806,0.574310,0.682157,0.680850,0.679868,0.675947


Completed ablation: without_matnet independent/gaussian | fallback priors: 0
Completed ablation: without_matnet independent/beta | fallback priors: 0
Completed ablation: without_matnet independent/dirichlet | fallback priors: 0
Completed ablation: without_matnet independent/ordinal_posterior | fallback priors: 0
Completed ablation: without_matnet spatial/gaussian | fallback priors: 0
Completed ablation: without_matnet spatial/beta | fallback priors: 0
Completed ablation: without_matnet spatial/dirichlet | fallback priors: 0
Completed ablation: without_matnet spatial/ordinal_posterior | fallback priors: 0


,scenario,n_tenders,n_original_polygons,avg_TAPOCB-NN_reward,median_TAPOCB-NN_reward,median_TAPOCB-NN_reward_ci_low,median_TAPOCB-NN_reward_ci_high,top_1,avg_top_1_reward,median_top_1_reward,median_top_1_reward_ci_low,median_top_1_reward_ci_high,avg_delta_TAPOCB-NN_from_top_1_units,median_delta_TAPOCB-NN_from_top_1_units,TAPOCB-NN_win_rate
0,independent/beta,16760,77,0.594299,0.654299,0.642074,0.667512,LKH-3,0.595690,0.655769,0.642102,0.666717,-0.001390,-0.001470,0.717930
1,independent/dirichlet,16760,77,0.603200,0.659171,0.644903,0.671620,LKH-3,0.604658,0.659323,0.644701,0.671740,-0.001458,-0.000152,0.711307
2,independent/gaussian,16760,77,0.600862,0.663499,0.642852,0.680678,LKH-3,0.602731,0.664802,0.644456,0.680326,-0.001869,-0.001303,0.721331
3,independent/ordinal_posterior,16760,77,0.584974,0.640430,0.616527,0.652836,LKH-3,0.589194,0.641333,0.620892,0.657211,-0.004219,-0.000903,0.696092
4,spatial/beta,16760,77,0.622213,0.690144,0.659017,0.705430,LKH-3,0.623383,0.688906,0.657682,0.705117,-0.001170,0.001238,0.735054
5,spatial/dirichlet,16760,77,0.619973,0.684528,0.654621,0.700639,LKH-3,0.620719,0.682254,0.653838,0.698349,-0.000745,0.002274,0.734517
6,spatial/gaussian,16760,77,0.630192,0.700954,0.684187,0.714527,LKH-3,0.630903,0.699999,0.683012,0.712628,-0.000712,0.000955,0.749672
7,spatial/ordinal_posterior,16760,77,0.619065,0.678007,0.650530,0.698461,LKH-3,0.623475,0.680806,0.657407,0.704194,-0.004409,-0.002799,0.718467


,scenario,median_TAPOCB-NN_reward,median_LKH-3_reward,median_MatNet_reward,median_Regression_reward,median_NeuralUCB_reward,median_NeuralTS_reward,median_NeuralRBMLE-PC_reward
0,independent/beta,0.654299,0.655769,0.572705,0.654207,0.654033,0.654199,0.650736
1,independent/dirichlet,0.659171,0.659323,0.568511,0.659219,0.657750,0.658409,0.656457
2,independent/gaussian,0.663499,0.664802,0.551736,0.663704,0.662703,0.661913,0.659481
3,independent/ordinal_posterior,0.640430,0.641333,0.533946,0.642859,0.643210,0.640484,0.638240
4,spatial/beta,0.690144,0.688906,0.605669,0.690699,0.688780,0.688958,0.683808
5,spatial/dirichlet,0.684528,0.682254,0.584601,0.684551,0.681798,0.682539,0.678034
6,spatial/gaussian,0.700954,0.699999,0.634624,0.701106,0.699504,0.699472,0.690812
7,spatial/ordinal_posterior,0.678007,0.680806,0.574310,0.682157,0.680850,0.679868,0.675947


... (12 outputs left out)

Completed ablation: no_injection independent/gaussian | fallback priors: 0
Completed ablation: no_injection independent/beta | fallback priors: 0
Completed ablation: no_injection independent/dirichlet | fallback priors: 0
Completed ablation: no_injection independent/ordinal_posterior | fallback priors: 0
Completed ablation: no_injection spatial/gaussian | fallback priors: 0
Completed ablation: no_injection spatial/beta | fallback priors: 0
Completed ablation: no_injection spatial/dirichlet | fallback priors: 0
Completed ablation: no_injection spatial/ordinal_posterior | fallback priors: 0


,scenario,n_tenders,n_original_polygons,avg_TAPOCB-NN_reward,median_TAPOCB-NN_reward,median_TAPOCB-NN_reward_ci_low,median_TAPOCB-NN_reward_ci_high,top_1,avg_top_1_reward,median_top_1_reward,median_top_1_reward_ci_low,median_top_1_reward_ci_high,avg_delta_TAPOCB-NN_from_top_1_units,median_delta_TAPOCB-NN_from_top_1_units,TAPOCB-NN_win_rate
0,independent/beta,16760,77,0.588079,0.653916,0.641011,0.667020,LKH-3,0.595690,0.655769,0.642102,0.666717,-0.007610,-0.001854,0.708562
1,independent/dirichlet,16760,77,0.596631,0.658623,0.643409,0.671162,LKH-3,0.604658,0.659323,0.644701,0.671740,-0.008026,-0.000700,0.702953
2,independent/gaussian,16760,77,0.594160,0.662314,0.640777,0.679363,LKH-3,0.602731,0.664802,0.644456,0.680326,-0.008571,-0.002488,0.713872
3,independent/ordinal_posterior,16760,77,0.573863,0.639327,0.612297,0.651887,LKH-3,0.589194,0.641333,0.620892,0.657211,-0.015331,-0.002005,0.678371
4,spatial/beta,16760,77,0.615921,0.688283,0.655271,0.704932,LKH-3,0.623383,0.688906,0.657682,0.705117,-0.007462,-0.000623,0.726342
5,spatial/dirichlet,16760,77,0.613051,0.682365,0.652599,0.700308,LKH-3,0.620719,0.682254,0.653838,0.698349,-0.007667,0.000111,0.724672
6,spatial/gaussian,16760,77,0.623241,0.699909,0.682557,0.714108,LKH-3,0.630903,0.699999,0.683012,0.712628,-0.007662,-0.000090,0.738335
7,spatial/ordinal_posterior,16760,77,0.608967,0.676406,0.648142,0.697894,LKH-3,0.623475,0.680806,0.657407,0.704194,-0.014507,-0.004400,0.702595


,scenario,median_TAPOCB-NN_reward,median_LKH-3_reward,median_MatNet_reward,median_Regression_reward,median_NeuralUCB_reward,median_NeuralTS_reward,median_NeuralRBMLE-PC_reward
0,independent/beta,0.653916,0.655769,0.572705,0.654207,0.654033,0.654199,0.650736
1,independent/dirichlet,0.658623,0.659323,0.568511,0.659219,0.657750,0.658409,0.656457
2,independent/gaussian,0.662314,0.664802,0.551736,0.663704,0.662703,0.661913,0.659481
3,independent/ordinal_posterior,0.639327,0.641333,0.533946,0.642859,0.643210,0.640484,0.638240
4,spatial/beta,0.688283,0.688906,0.605669,0.690699,0.688780,0.688958,0.683808
5,spatial/dirichlet,0.682365,0.682254,0.584601,0.684551,0.681798,0.682539,0.678034
6,spatial/gaussian,0.699909,0.699999,0.634624,0.701106,0.699504,0.699472,0.690812
7,spatial/ordinal_posterior,0.676406,0.680806,0.574310,0.682157,0.680850,0.679868,0.675947


,contour,variant,scenario,n_tenders,mean_delta_reward,median_delta_reward,ablation_pairwise_win_rate
0,no_warmstart,no_warmstart,independent/beta,16760,-0.025245,1.908154e-05,0.731414
1,no_warmstart,no_warmstart,independent/dirichlet,16760,-0.025520,1.947915e-05,0.727924
2,no_warmstart,no_warmstart,independent/gaussian,16760,-0.022391,2.000510e-05,0.736754
3,no_warmstart,no_warmstart,independent/ordinal_posterior,16760,-0.017110,1.822343e-05,0.769690
4,no_warmstart,no_warmstart,spatial/beta,16760,-0.025544,2.238185e-05,0.736635
5,no_warmstart,no_warmstart,spatial/dirichlet,16760,-0.025170,2.079541e-05,0.736724
6,no_warmstart,no_warmstart,spatial/gaussian,16760,-0.023827,2.276682e-05,0.730012
7,no_warmstart,no_warmstart,spatial/ordinal_posterior,16760,-0.016917,2.171736e-05,0.779803
8,drop_arm,without_matnet,independent/beta,16760,0.006670,0.000000e+00,0.535919
9,drop_arm,without_matnet,independent/dirichlet,16760,0.006897,9.139789e-08,0.556533
